# Session 7: Data Cleaning with Pandas

**Course:** Python for Data Engineering  
**Phase 2:** Data Handling & Transformation

**What we'll cover:**
- Handling missing values (nulls)
- Data type conversions
- String cleaning and normalization
- Filtering rows
- Sorting data

**Data files:** `data/sales_messy.csv`, `data/employees_messy.csv`

**Note:** This session is demo-heavy. Follow along by running each cell — the goal is to learn the cleaning patterns, not write everything from scratch.

In [ ]:
import pandas as pd
import numpy as np

---

## 1. Why Data Cleaning Matters

Raw data is almost never clean. In real pipelines you'll deal with:
- Missing values (empty cells, `N/A`, `null`)
- Wrong types (numbers stored as strings, dates as text)
- Inconsistent formatting (`NORTH`, `north`, ` North `)
- Invalid entries (negative quantities, `free` instead of a price)

Data cleaning is typically 60-80% of a data engineer's work. pandas gives you the tools to do it efficiently.

---

## 2. Load and Inspect the Messy Data

Always start by looking at what you have before cleaning anything.

In [ ]:
# Load the messy sales data

df = pd.read_csv("data/sales_messy.csv")
df

In [ ]:
# Quick inspection — check types, nulls, shape

print(f"Shape: {df.shape}")
print()
df.info()

In [ ]:
# Check for nulls per column

print("Null counts:")
print(df.isnull().sum())
print(f"\nTotal nulls: {df.isnull().sum().sum()}")

Problems we can see:
- `product` has a missing value (empty cell)
- `quantity` has `ten` instead of a number — pandas read it as string (object)
- `unit_price` has `free` instead of a dollar amount
- `region` has a missing value
- `quantity` has a negative value (-2)

---

## 3. Handling Missing Values

pandas uses `NaN` (Not a Number) to represent missing data. You have three options:

| Method | When to use |
|--------|------------|
| `dropna()` | Row is useless without the missing field |
| `fillna(value)` | You have a sensible default |
| `fillna(method='ffill')` | Fill with previous value (time series) |

In [ ]:
# Detect nulls

print("Is each cell null?")
print(df.isnull())
print("\nWhich rows have ANY null?")
print(df[df.isnull().any(axis=1)])

In [ ]:
# Option 1: Drop rows with nulls

df_dropped = df.dropna()
print(f"Before: {len(df)} rows")
print(f"After dropna(): {len(df_dropped)} rows")
print(f"Dropped: {len(df) - len(df_dropped)} rows")

In [ ]:
# Option 2: Drop rows only if specific columns are null
# product is critical — can't have an order without a product

df_dropped_product = df.dropna(subset=["product"])
print(f"Dropped rows missing product: {len(df) - len(df_dropped_product)}")

In [ ]:
# Option 3: Fill nulls with a default value
# region is less critical — fill with "unknown"

df_filled = df.copy()
df_filled["region"] = df_filled["region"].fillna("unknown")
print(df_filled[["product", "region"]])

In [ ]:
# fillna with different values per column

df_filled2 = df.fillna({
    "product": "Unknown Product",
    "region": "unknown",
})
df_filled2

**In practice:** For data pipelines, you usually drop rows where key columns are null and fill non-critical columns with defaults. The decision depends on your business rules.

---

## 4. Data Type Conversions

When pandas reads a CSV, it guesses types. Sometimes it gets them wrong — especially when a column has mixed values like `5` and `ten`.

In [ ]:
# Check current types

print(df.dtypes)
print()
print("Notice: quantity is 'object' (string) because of 'ten'")
print("Notice: unit_price is 'object' because of '$' and 'free'")

In [ ]:
# pd.to_numeric — converts strings to numbers, with error handling

clean = df.copy()

# errors='coerce' turns unparseable values into NaN instead of crashing
clean["quantity"] = pd.to_numeric(clean["quantity"], errors="coerce")

print("Quantity after to_numeric:")
print(clean["quantity"])
print(f"\nType: {clean['quantity'].dtype}")
print(f"NaN count: {clean['quantity'].isna().sum()}")

In [ ]:
# Clean unit_price — remove '$' first, then convert

clean["unit_price"] = clean["unit_price"].str.replace("$", "", regex=False)
clean["unit_price"] = pd.to_numeric(clean["unit_price"], errors="coerce")

print("unit_price after cleaning:")
print(clean["unit_price"])
print(f"\nType: {clean['unit_price'].dtype}")

In [ ]:
# pd.to_datetime — convert strings to datetime objects

clean["date"] = pd.to_datetime(clean["date"], errors="coerce")

print(f"Date type: {clean['date'].dtype}")
print(clean["date"])

### Type conversion reference

| Function | What it does | Handles bad values? |
|----------|-------------|--------------------|
| `pd.to_numeric(col, errors='coerce')` | String → number | Yes, sets to NaN |
| `pd.to_datetime(col, errors='coerce')` | String → datetime | Yes, sets to NaT |
| `col.astype(float)` | Force type conversion | No — crashes on bad data |
| `col.astype(str)` | Convert to string | Always works |

**Rule of thumb:** Use `pd.to_numeric`/`pd.to_datetime` with `errors='coerce'` when data might be dirty. Use `.astype()` only when you're sure the data is clean.

---

## 5. String Cleaning

String columns in raw data are almost always messy — extra spaces, inconsistent casing, typos. The `.str` accessor gives you all the tools.

In [ ]:
# Load the messy employees to demo string cleaning

emp = pd.read_csv("data/employees_messy.csv")
emp

In [ ]:
# strip() — remove leading/trailing whitespace

print("Before strip:")
print(emp["name"].tolist())

emp["name"] = emp["name"].str.strip()

print("\nAfter strip:")
print(emp["name"].tolist())

In [ ]:
# Casing — title(), lower(), upper()

emp["name"] = emp["name"].str.title()
emp["department"] = emp["department"].str.strip().str.lower()

print("Names (title case):")
print(emp["name"].tolist())
print("\nDepartments (lowercase):")
print(emp["department"].tolist())

In [ ]:
# replace() — fix known bad values

# Clean the salary column — remove $ and commas
emp["salary"] = emp["salary"].str.replace("$", "", regex=False)
emp["salary"] = emp["salary"].str.replace(",", "", regex=False)
emp["salary"] = pd.to_numeric(emp["salary"], errors="coerce")

print("Cleaned salary:")
print(emp[["name", "salary"]])

In [ ]:
# contains() — find rows matching a pattern

# Find employees with 'unknown' department
unknown_dept = emp[emp["department"].str.contains("unknown", na=False)]
print("Unknown department:")
print(unknown_dept[["name", "department"]])

### String cleaning reference

| Method | What it does |
|--------|-------------|
| `.str.strip()` | Remove leading/trailing whitespace |
| `.str.lower()` | Convert to lowercase |
| `.str.upper()` | Convert to uppercase |
| `.str.title()` | Convert to Title Case |
| `.str.replace(old, new)` | Replace substrings |
| `.str.contains(pattern)` | Check if string contains pattern |
| `.str.startswith(prefix)` | Check prefix |
| `.str.len()` | Length of each string |

---

## 6. Replacing Values

Sometimes you need to map bad values to correct ones, or standardize categories.

In [ ]:
# replace() on a DataFrame — fix specific values

emp_clean = emp.copy()

# Replace 'unknown' department with NaN
emp_clean["department"] = emp_clean["department"].replace("unknown", np.nan)

# Replace 'N/A' age with NaN
emp_clean["age"] = emp_clean["age"].replace("N/A", np.nan)
emp_clean["age"] = pd.to_numeric(emp_clean["age"], errors="coerce")

print(emp_clean[["name", "age", "department"]])

In [ ]:
# map() — apply a mapping dict to a column
# useful for standardizing categories

region_map = {
    "n": "north",
    "s": "south",
    "e": "east",
    "w": "west",
    "north": "north",
    "south": "south",
    "east": "east",
    "west": "west",
}

# Demo with the sales data
sales = pd.read_csv("data/sales.csv")
sales["region"] = sales["region"].str.strip().str.lower().map(region_map)
print(sales[["product", "region"]])

---

## 7. Filtering Rows

Filtering is how you select rows that meet a condition. This replaces `if` statements inside `for` loops.

In [ ]:
# Reload clean sales data for filtering demos

df = pd.read_csv("data/sales.csv")
df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False).astype(float)
df["product"] = df["product"].str.strip().str.title()
df["region"] = df["region"].str.strip().str.lower()
df["total"] = df["quantity"] * df["unit_price"]
df

In [ ]:
# Single condition

positive_qty = df[df["quantity"] > 0]
print(f"Positive quantity: {len(positive_qty)} of {len(df)} rows")
print(positive_qty)

In [ ]:
# Multiple conditions — use & (and), | (or)
# each condition needs parentheses

# North region AND quantity > 1
north_bulk = df[(df["region"] == "north") & (df["quantity"] > 1)]
print("North region, quantity > 1:")
print(north_bulk)

In [ ]:
# isin() — filter by a list of values

laptops_keyboards = df[df["product"].isin(["Laptop", "Keyboard"])]
print("Laptops and Keyboards:")
print(laptops_keyboards)

In [ ]:
# Negation — use ~ to invert

not_north = df[~(df["region"] == "north")]
print("Not north:")
print(not_north)

In [ ]:
# query() — alternative syntax, easier to read for complex conditions

result = df.query("region == 'north' and quantity > 1")
print(result)

### Filtering reference

| Pattern | Syntax |
|---------|--------|
| Single condition | `df[df["col"] > 5]` |
| AND | `df[(cond1) & (cond2)]` |
| OR | `df[(cond1) \| (cond2)]` |
| NOT | `df[~(condition)]` |
| Value in list | `df[df["col"].isin(["a", "b"])]` |
| String contains | `df[df["col"].str.contains("text")]` |
| Query style | `df.query("col > 5 and name == 'x'")` |

---

## 8. Sorting Data

Sorting is straightforward but important for reports and debugging.

In [ ]:
# Sort by a single column

by_total = df.sort_values("total", ascending=False)
print("By total (highest first):")
print(by_total[["product", "quantity", "total"]])

In [ ]:
# Sort by multiple columns

by_region_total = df.sort_values(["region", "total"], ascending=[True, False])
print("By region (A-Z), then total (highest first):")
print(by_region_total[["region", "product", "total"]])

In [ ]:
# nlargest / nsmallest — quick way to get top/bottom N

print("Top 3 by total:")
print(df.nlargest(3, "total")[["product", "total"]])

print("\nBottom 2 by quantity:")
print(df.nsmallest(2, "quantity")[["product", "quantity"]])

---

## 9. Duplicates

Duplicate rows are common when data comes from multiple sources or when upstream systems have bugs.

In [ ]:
# Check for duplicates

print(f"Duplicate rows: {df.duplicated().sum()}")

# Check for duplicate products (maybe expected)
print(f"Duplicate products: {df['product'].duplicated().sum()}")
print(f"\nProduct value counts:")
print(df["product"].value_counts())

In [ ]:
# Remove duplicates — keep first occurrence

# Simulate duplicates
df_dupes = pd.concat([df, df.iloc[:2]], ignore_index=True)
print(f"With dupes: {len(df_dupes)} rows")

df_deduped = df_dupes.drop_duplicates()
print(f"After dedup: {len(df_deduped)} rows")

# Drop duplicates based on specific columns
df_deduped2 = df_dupes.drop_duplicates(subset=["date", "product", "region"])
print(f"Dedup on date+product+region: {len(df_deduped2)} rows")

---

## 10. Putting It All Together — A Cleaning Pipeline

In practice, you chain all these steps into a cleaning function. Here's a realistic example.

In [ ]:
def clean_sales_data(filepath):
    """Clean raw sales CSV and return a clean DataFrame."""
    # Extract
    df = pd.read_csv(filepath)
    rows_before = len(df)

    # Drop rows missing critical fields
    df = df.dropna(subset=["product"])

    # Fill non-critical nulls
    df["region"] = df["region"].fillna("unknown")

    # Clean strings
    df["product"] = df["product"].str.strip().str.title()
    df["region"] = df["region"].str.strip().str.lower()

    # Fix types
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False)
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    # Drop rows where conversion failed
    df = df.dropna(subset=["quantity", "unit_price"])

    # Filter invalid data
    df = df[df["quantity"] > 0]

    # Convert quantity to int
    df["quantity"] = df["quantity"].astype(int)

    # Add computed columns
    df["total"] = df["quantity"] * df["unit_price"]

    # Remove duplicates
    df = df.drop_duplicates()

    # Sort
    df = df.sort_values("date").reset_index(drop=True)

    rows_after = len(df)
    print(f"Cleaned: {rows_before} → {rows_after} rows ({rows_before - rows_after} removed)")

    return df

In [ ]:
# Run the cleaning pipeline

clean_sales = clean_sales_data("data/sales_messy.csv")
print()
clean_sales

In [ ]:
# Verify the output

print("Types:")
print(clean_sales.dtypes)
print(f"\nNulls: {clean_sales.isnull().sum().sum()}")
print(f"Negative quantities: {(clean_sales['quantity'] <= 0).sum()}")

---

## Lab: Clean the Messy Employee Data

Apply everything from this session to clean `data/employees_messy.csv`.

**Steps:**
1. Load the file and inspect it (`info()`, `isnull().sum()`, look at the raw data)
2. Clean names — strip whitespace, title case
3. Clean department — strip, lowercase; replace 'unknown' with NaN
4. Clean salary — remove `$`, convert to numeric
5. Clean age — handle `N/A`, convert to numeric
6. Drop rows where name is null
7. Fill missing departments with `'unassigned'`
8. Convert `start_date` to datetime
9. Sort by name
10. Print the clean DataFrame and a summary (row count, avg salary, null counts)

In [ ]:
import pandas as pd
import numpy as np

# Your code here

---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| Nulls | `isnull()`, `dropna()`, `fillna()` — detect and handle missing data |
| Types | `pd.to_numeric()`, `pd.to_datetime()` with `errors='coerce'` for dirty data |
| Strings | `.str.strip()`, `.str.lower()`, `.str.replace()` — clean and normalize |
| Replacing | `.replace()` for specific values, `.map()` for category mappings |
| Filtering | `df[condition]`, `&` for AND, `|` for OR, `~` for NOT |
| Sorting | `sort_values()`, `nlargest()`, `nsmallest()` |
| Duplicates | `duplicated()`, `drop_duplicates()` |

**Key patterns:**
- Always inspect data before cleaning — `info()`, `isnull().sum()`, `value_counts()`
- Use `errors='coerce'` to convert safely — bad values become NaN
- Chain cleaning steps into a reusable function
- Decide per-column: drop nulls or fill with defaults

**Next session:** Data Aggregation & Joins — groupby operations and merging datasets.